# 🧪 Testes Automatizados — EcoPonto BH

**TP5 — Engenharia de Software**

Este notebook executa **18 testes automatizados** cobrindo os 6 casos de uso do sistema:

| # | Caso de Uso | Endpoint |
|---|-------------|----------|
| UC1 | Consultar pontos de coleta | `GET /api/pontos` |
| UC2 | Pesquisar pontos de coleta | `GET /api/pontos` + filtro |
| UC3 | Visualizar detalhes do ponto | `GET /api/pontos/:id` |
| UC4 | Cadastrar ponto de coleta | `POST /api/pontos` |
| UC5 | Editar ponto de coleta | `PUT /api/pontos/:id` |
| UC6 | Excluir ponto de coleta | `DELETE /api/pontos/:id` |

Cada UC possui **3 testes**: fluxo principal, edge case e fluxo de exceção.

> ⚠️ **Pré-requisito:** o backend deve estar rodando em `http://localhost:3001`.
> Execute `npm run dev` dentro da pasta `/backend` antes de rodar este notebook.

## ⚙️ Setup

In [ ]:
import requests
import json
import sys

BASE_URL = "http://localhost:3001/api/pontos"

# Contadores globais
resultados = {"passou": 0, "falhou": 0}

def ok(nome):
    resultados["passou"] += 1
    print(f"  ✅ PASSOU — {nome}")

def falhou(nome, motivo):
    resultados["falhou"] += 1
    print(f"  ❌ FALHOU — {nome}: {motivo}")

def executar_teste(nome, fn):
    print(f"\n🔹 {nome}")
    try:
        fn()
        ok(nome)
    except AssertionError as e:
        falhou(nome, str(e))
    except Exception as e:
        falhou(nome, f"Erro inesperado: {e}")

# Verifica conectividade com o backend
try:
    r = requests.get(BASE_URL, timeout=3)
    print(f"✅ Backend acessível em {BASE_URL} (status {r.status_code})")
except Exception as e:
    print(f"❌ Backend inacessível: {e}")
    print("   Execute 'npm run dev' na pasta /backend e reinicie o notebook.")
    sys.exit(1)

---
## UC1 — Consultar pontos de coleta
`GET /api/pontos`

In [ ]:
print("=" * 55)
print("UC1 — Consultar pontos de coleta")
print("=" * 55)

# ── Fluxo Principal ──────────────────────────────────────
def uc1_fluxo_principal():
    """GET /api/pontos retorna 200 e uma lista JSON."""
    r = requests.get(BASE_URL)
    assert r.status_code == 200, f"Status esperado 200, recebido {r.status_code}"
    dados = r.json()
    assert isinstance(dados, list), "Resposta deve ser uma lista"
    print(f"     → {len(dados)} ponto(s) retornado(s)")

executar_teste("UC1-T1 | Fluxo Principal: listar todos os pontos", uc1_fluxo_principal)

# ── Edge Case ────────────────────────────────────────────
def uc1_edge_case():
    """Campo 'residuos' de cada ponto é uma lista Python (não string JSON)."""
    r = requests.get(BASE_URL)
    dados = r.json()
    if len(dados) == 0:
        # Cria um ponto temporário para validar o campo
        payload = {"nome": "Teste EC", "endereco": "Rua X", "bairro": "Centro",
                   "residuos": ["Papel", "Vidro"], "latitude": -19.92, "longitude": -43.94}
        criado = requests.post(BASE_URL, json=payload).json()
        dados = requests.get(BASE_URL).json()
        requests.delete(f"{BASE_URL}/{criado['id']}")
    for p in dados:
        assert isinstance(p["residuos"], list), \
            f"'residuos' deveria ser lista, mas é {type(p['residuos']).__name__} no ponto id={p['id']}"
    print(f"     → {len(dados)} ponto(s) verificado(s): campo 'residuos' é lista em todos")

executar_teste("UC1-T2 | Edge Case: campo 'residuos' deserializado como lista", uc1_edge_case)

# ── Fluxo de Exceção ─────────────────────────────────────
def uc1_fluxo_excecao():
    """Requisição para endpoint inexistente retorna 404 (não 200)."""
    r = requests.get(BASE_URL + "/rota-inexistente-xyz")
    assert r.status_code == 404, \
        f"Esperado 404 para rota inválida, recebido {r.status_code}"
    print(f"     → Status 404 confirmado para rota inválida")

executar_teste("UC1-T3 | Exceção: rota inexistente retorna 404", uc1_fluxo_excecao)

---
## UC2 — Pesquisar pontos de coleta
Filtragem client-side por bairro e tipo de resíduo

In [ ]:
print("=" * 55)
print("UC2 — Pesquisar pontos de coleta")
print("=" * 55)

# Dados de apoio para UC2
_uc2_criados = []

def _criar_pontos_uc2():
    pontos = [
        {"nome": "Ecoponto Savassi", "endereco": "Av. do Contorno, 100", "bairro": "Savassi",
         "residuos": ["Papel", "Plástico"], "latitude": -19.934, "longitude": -43.931},
        {"nome": "Ecoponto Centro", "endereco": "Rua da Bahia, 50", "bairro": "Centro",
         "residuos": ["Vidro", "Metal"], "latitude": -19.917, "longitude": -43.934},
        {"nome": "Ecoponto Lourdes", "endereco": "Rua Cláudio Manoel, 20", "bairro": "Lourdes",
         "residuos": ["Papel", "Eletrônico"], "latitude": -19.938, "longitude": -43.936},
    ]
    for p in pontos:
        r = requests.post(BASE_URL, json=p)
        _uc2_criados.append(r.json()["id"])

def _limpar_pontos_uc2():
    for pid in _uc2_criados:
        requests.delete(f"{BASE_URL}/{pid}")
    _uc2_criados.clear()

_criar_pontos_uc2()

# ── Fluxo Principal ──────────────────────────────────────
def uc2_fluxo_principal():
    """Filtrar pontos por bairro retorna apenas os do bairro pesquisado."""
    todos = requests.get(BASE_URL).json()
    bairro_alvo = "Savassi"
    filtrados = [p for p in todos if p["bairro"].lower() == bairro_alvo.lower()]
    assert len(filtrados) >= 1, f"Nenhum ponto encontrado no bairro '{bairro_alvo}'"
    for p in filtrados:
        assert p["bairro"].lower() == bairro_alvo.lower(), \
            f"Ponto id={p['id']} retornou bairro '{p['bairro']}', esperado '{bairro_alvo}'"
    print(f"     → {len(filtrados)} ponto(s) no bairro '{bairro_alvo}'")

executar_teste("UC2-T1 | Fluxo Principal: filtrar por bairro", uc2_fluxo_principal)

# ── Edge Case ────────────────────────────────────────────
def uc2_edge_case():
    """Busca por tipo de resíduo retorna pontos que aceitam aquele material."""
    todos = requests.get(BASE_URL).json()
    residuo_alvo = "Papel"
    filtrados = [p for p in todos if residuo_alvo in p["residuos"]]
    assert len(filtrados) >= 1, f"Nenhum ponto aceita '{residuo_alvo}'"
    for p in filtrados:
        assert residuo_alvo in p["residuos"], \
            f"Ponto id={p['id']} não aceita '{residuo_alvo}'"
    print(f"     → {len(filtrados)} ponto(s) aceitam '{residuo_alvo}'")

executar_teste("UC2-T2 | Edge Case: filtrar por tipo de resíduo", uc2_edge_case)

# ── Fluxo de Exceção ─────────────────────────────────────
def uc2_fluxo_excecao():
    """Busca por bairro inexistente retorna lista vazia (sem erro)."""
    todos = requests.get(BASE_URL).json()
    bairro_fantasma = "BairroQueNaoExiste123"
    filtrados = [p for p in todos if p["bairro"].lower() == bairro_fantasma.lower()]
    assert isinstance(filtrados, list), "Resultado deve ser uma lista"
    assert len(filtrados) == 0, f"Esperado 0 resultados, encontrado {len(filtrados)}"
    print(f"     → Busca por '{bairro_fantasma}' retornou lista vazia corretamente")

executar_teste("UC2-T3 | Exceção: busca por bairro inexistente retorna lista vazia", uc2_fluxo_excecao)

_limpar_pontos_uc2()

---
## UC3 — Visualizar detalhes do ponto
`GET /api/pontos/:id`

In [ ]:
print("=" * 55)
print("UC3 — Visualizar detalhes do ponto")
print("=" * 55)

# Cria ponto de apoio
_uc3_payload = {
    "nome": "Ecoponto Belvedere",
    "endereco": "Av. Raja Gabaglia, 400",
    "bairro": "Belvedere",
    "descricao": "Ponto com coleta de eletrônicos",
    "residuos": ["Eletrônico", "Pilha", "Bateria"],
    "latitude": -19.962,
    "longitude": -43.962
}
_uc3_ponto = requests.post(BASE_URL, json=_uc3_payload).json()
_uc3_id = _uc3_ponto["id"]
print(f"  [setup] Ponto de apoio criado com id={_uc3_id}")

# ── Fluxo Principal ──────────────────────────────────────
def uc3_fluxo_principal():
    """GET /:id retorna 200 e todos os campos do ponto."""
    r = requests.get(f"{BASE_URL}/{_uc3_id}")
    assert r.status_code == 200, f"Status esperado 200, recebido {r.status_code}"
    p = r.json()
    campos_obrigatorios = ["id", "nome", "endereco", "bairro", "residuos", "latitude", "longitude"]
    for campo in campos_obrigatorios:
        assert campo in p, f"Campo '{campo}' ausente na resposta"
    assert p["id"] == _uc3_id, f"ID esperado {_uc3_id}, recebido {p['id']}"
    print(f"     → Ponto '{p['nome']}' retornado com todos os campos obrigatórios")

executar_teste("UC3-T1 | Fluxo Principal: buscar ponto por ID válido", uc3_fluxo_principal)

# ── Edge Case ────────────────────────────────────────────
def uc3_edge_case():
    """Ponto com múltiplos resíduos e descricao retorna todos os campos corretamente."""
    r = requests.get(f"{BASE_URL}/{_uc3_id}")
    p = r.json()
    assert isinstance(p["residuos"], list), "'residuos' deve ser lista"
    assert len(p["residuos"]) == 3, f"Esperado 3 resíduos, retornado {len(p['residuos'])}"
    assert p["descricao"] is not None, "'descricao' deve estar presente"
    assert isinstance(p["latitude"], float), "'latitude' deve ser float"
    assert isinstance(p["longitude"], float), "'longitude' deve ser float"
    print(f"     → residuos={p['residuos']}, descricao='{p['descricao']}'")

executar_teste("UC3-T2 | Edge Case: ponto com múltiplos resíduos e descrição", uc3_edge_case)

# ── Fluxo de Exceção ─────────────────────────────────────
def uc3_fluxo_excecao():
    """GET /:id com ID inexistente retorna 404."""
    id_invalido = 999999
    r = requests.get(f"{BASE_URL}/{id_invalido}")
    assert r.status_code == 404, \
        f"Esperado 404 para id={id_invalido}, recebido {r.status_code}"
    corpo = r.json()
    assert "error" in corpo, "Corpo da resposta deve conter campo 'error'"
    print(f"     → Status 404 e mensagem: '{corpo['error']}'")

executar_teste("UC3-T3 | Exceção: ID inexistente retorna 404 com mensagem de erro", uc3_fluxo_excecao)

# Limpeza
requests.delete(f"{BASE_URL}/{_uc3_id}")
print(f"  [teardown] Ponto id={_uc3_id} removido")

---
## UC4 — Cadastrar ponto de coleta
`POST /api/pontos`

In [ ]:
print("=" * 55)
print("UC4 — Cadastrar ponto de coleta")
print("=" * 55)

_uc4_ids = []  # coleta IDs criados para limpeza

# ── Fluxo Principal ──────────────────────────────────────
def uc4_fluxo_principal():
    """POST com todos os campos retorna 201 e o ponto criado com ID gerado."""
    payload = {
        "nome": "Ecoponto Funcionários",
        "endereco": "Rua Maranhão, 200",
        "bairro": "Funcionários",
        "descricao": "Coleta seletiva completa",
        "residuos": ["Papel", "Plástico", "Vidro", "Metal"],
        "latitude": -19.929,
        "longitude": -43.933
    }
    r = requests.post(BASE_URL, json=payload)
    assert r.status_code == 201, f"Status esperado 201, recebido {r.status_code}"
    p = r.json()
    assert "id" in p and p["id"] is not None, "Ponto criado deve ter 'id'"
    assert p["nome"] == payload["nome"], f"Nome retornado: '{p['nome']}'"
    assert p["residuos"] == payload["residuos"], f"Resíduos retornados: {p['residuos']}"
    _uc4_ids.append(p["id"])
    print(f"     → Ponto criado com id={p['id']}, nome='{p['nome']}'")

executar_teste("UC4-T1 | Fluxo Principal: cadastrar ponto com todos os campos", uc4_fluxo_principal)

# ── Edge Case ────────────────────────────────────────────
def uc4_edge_case():
    """POST sem campos opcionais (descricao e residuos) ainda cria o ponto."""
    payload = {
        "nome": "Ecoponto Mínimo",
        "endereco": "Rua Teste, 1",
        "bairro": "Pampulha",
        "latitude": -19.868,
        "longitude": -43.972
        # sem 'descricao' e sem 'residuos'
    }
    r = requests.post(BASE_URL, json=payload)
    assert r.status_code == 201, f"Status esperado 201, recebido {r.status_code}"
    p = r.json()
    assert p["descricao"] is None, f"'descricao' deveria ser null, foi '{p['descricao']}'"
    assert p["residuos"] == [], f"'residuos' deveria ser [], foi {p['residuos']}"
    _uc4_ids.append(p["id"])
    print(f"     → Ponto mínimo criado com id={p['id']}: descricao=null, residuos=[]")

executar_teste("UC4-T2 | Edge Case: cadastrar ponto sem campos opcionais", uc4_edge_case)

# ── Fluxo de Exceção ─────────────────────────────────────
def uc4_fluxo_excecao():
    """POST sem campos obrigatórios retorna 400 com mensagem de erro."""
    payload_invalido = {
        "descricao": "Ponto sem campos obrigatórios",
        "residuos": ["Papel"]
        # sem nome, endereco, bairro, latitude, longitude
    }
    r = requests.post(BASE_URL, json=payload_invalido)
    assert r.status_code == 400, \
        f"Esperado 400 para payload inválido, recebido {r.status_code}"
    corpo = r.json()
    assert "error" in corpo, "Resposta 400 deve conter campo 'error'"
    print(f"     → Status 400 com mensagem: '{corpo['error']}'")

executar_teste("UC4-T3 | Exceção: POST sem campos obrigatórios retorna 400", uc4_fluxo_excecao)

# Limpeza
for pid in _uc4_ids:
    requests.delete(f"{BASE_URL}/{pid}")
print(f"  [teardown] {len(_uc4_ids)} ponto(s) removido(s)")

---
## UC5 — Editar ponto de coleta
`PUT /api/pontos/:id`

In [ ]:
print("=" * 55)
print("UC5 — Editar ponto de coleta")
print("=" * 55)

# Cria ponto de apoio
_uc5_payload_inicial = {
    "nome": "Ecoponto Original",
    "endereco": "Rua Original, 10",
    "bairro": "Santa Efigênia",
    "descricao": "Descrição original",
    "residuos": ["Papel"],
    "latitude": -19.923,
    "longitude": -43.921
}
_uc5_ponto = requests.post(BASE_URL, json=_uc5_payload_inicial).json()
_uc5_id = _uc5_ponto["id"]
print(f"  [setup] Ponto de apoio criado com id={_uc5_id}")

# ── Fluxo Principal ──────────────────────────────────────
def uc5_fluxo_principal():
    """PUT /:id atualiza campos enviados e retorna o ponto atualizado."""
    atualizacao = {
        "nome": "Ecoponto Atualizado",
        "residuos": ["Papel", "Plástico", "Vidro"]
    }
    r = requests.put(f"{BASE_URL}/{_uc5_id}", json=atualizacao)
    assert r.status_code == 200, f"Status esperado 200, recebido {r.status_code}"
    p = r.json()
    assert p["nome"] == atualizacao["nome"], \
        f"Nome esperado '{atualizacao['nome']}', recebido '{p['nome']}'"
    assert p["residuos"] == atualizacao["residuos"], \
        f"Resíduos esperados {atualizacao['residuos']}, recebidos {p['residuos']}"
    print(f"     → nome='{p['nome']}', residuos={p['residuos']}")

executar_teste("UC5-T1 | Fluxo Principal: atualizar nome e resíduos", uc5_fluxo_principal)

# ── Edge Case ────────────────────────────────────────────
def uc5_edge_case():
    """PUT com apenas um campo não sobrescreve os demais campos existentes."""
    # Busca estado atual antes da edição parcial
    antes = requests.get(f"{BASE_URL}/{_uc5_id}").json()
    novo_bairro = "Serra"
    r = requests.put(f"{BASE_URL}/{_uc5_id}", json={"bairro": novo_bairro})
    assert r.status_code == 200, f"Status esperado 200, recebido {r.status_code}"
    depois = r.json()
    assert depois["bairro"] == novo_bairro, \
        f"Bairro esperado '{novo_bairro}', recebido '{depois['bairro']}'"
    assert depois["nome"] == antes["nome"], \
        f"'nome' não deveria mudar: antes='{antes['nome']}', depois='{depois['nome']}'"
    assert depois["endereco"] == antes["endereco"], \
        f"'endereco' não deveria mudar"
    print(f"     → bairro atualizado para '{novo_bairro}', demais campos preservados")

executar_teste("UC5-T2 | Edge Case: edição parcial preserva campos não enviados", uc5_edge_case)

# ── Fluxo de Exceção ─────────────────────────────────────
def uc5_fluxo_excecao():
    """PUT com ID inexistente retorna erro (500 do Prisma — registro não encontrado)."""
    id_invalido = 999999
    r = requests.put(f"{BASE_URL}/{id_invalido}", json={"nome": "Fantasma"})
    assert r.status_code in (404, 500), \
        f"Esperado 404 ou 500 para id inexistente, recebido {r.status_code}"
    print(f"     → Status {r.status_code} para PUT em id={id_invalido} (não encontrado)")

executar_teste("UC5-T3 | Exceção: editar ID inexistente retorna erro", uc5_fluxo_excecao)

# Limpeza
requests.delete(f"{BASE_URL}/{_uc5_id}")
print(f"  [teardown] Ponto id={_uc5_id} removido")

---
## UC6 — Excluir ponto de coleta
`DELETE /api/pontos/:id`

In [ ]:
print("=" * 55)
print("UC6 — Excluir ponto de coleta")
print("=" * 55)

# ── Fluxo Principal ──────────────────────────────────────
def uc6_fluxo_principal():
    """DELETE /:id retorna 204 e o ponto é removido do banco."""
    # Cria ponto para deletar
    criado = requests.post(BASE_URL, json={
        "nome": "Ecoponto para Deletar",
        "endereco": "Rua Delete, 0",
        "bairro": "Venda Nova",
        "latitude": -19.831,
        "longitude": -43.981
    }).json()
    pid = criado["id"]

    # Deleta
    r = requests.delete(f"{BASE_URL}/{pid}")
    assert r.status_code == 204, f"Status esperado 204, recebido {r.status_code}"

    # Confirma que não existe mais
    r_check = requests.get(f"{BASE_URL}/{pid}")
    assert r_check.status_code == 404, \
        f"Após DELETE, GET deveria retornar 404, recebido {r_check.status_code}"
    print(f"     → Ponto id={pid} deletado (204) e confirmado ausente (404)")

executar_teste("UC6-T1 | Fluxo Principal: excluir ponto existente", uc6_fluxo_principal)

# ── Edge Case ────────────────────────────────────────────
def uc6_edge_case():
    """Após DELETE, o ponto não aparece mais na listagem geral."""
    criado = requests.post(BASE_URL, json={
        "nome": "Ecoponto Efêmero",
        "endereco": "Rua Efêmera, 1",
        "bairro": "Barreiro",
        "latitude": -19.981,
        "longitude": -44.016
    }).json()
    pid = criado["id"]

    requests.delete(f"{BASE_URL}/{pid}")

    todos = requests.get(BASE_URL).json()
    ids = [p["id"] for p in todos]
    assert pid not in ids, \
        f"Ponto id={pid} ainda aparece na listagem após DELETE"
    print(f"     → Ponto id={pid} ausente na listagem após exclusão ({len(todos)} ponto(s) restante(s))")

executar_teste("UC6-T2 | Edge Case: ponto deletado não aparece na listagem", uc6_edge_case)

# ── Fluxo de Exceção ─────────────────────────────────────
def uc6_fluxo_excecao():
    """DELETE com ID inexistente retorna erro (500 do Prisma — P2025)."""
    id_invalido = 999999
    r = requests.delete(f"{BASE_URL}/{id_invalido}")
    assert r.status_code in (404, 500), \
        f"Esperado 404 ou 500 para DELETE em id inexistente, recebido {r.status_code}"
    print(f"     → Status {r.status_code} ao tentar deletar id={id_invalido} (não encontrado)")

executar_teste("UC6-T3 | Exceção: deletar ID inexistente retorna erro", uc6_fluxo_excecao)

---
## 📊 Resultado Final

In [ ]:
total = resultados["passou"] + resultados["falhou"]
print("\n" + "=" * 55)
print("RESULTADO FINAL")
print("=" * 55)
print(f"  Total de testes : {total}")
print(f"  ✅ Passaram      : {resultados['passou']}")
print(f"  ❌ Falharam      : {resultados['falhou']}")
print("=" * 55)
if resultados["falhou"] == 0:
    print("\n🎉 Todos os testes passaram!")
else:
    pct = (resultados['passou'] / total * 100) if total else 0
    print(f"\n⚠️  Taxa de sucesso: {pct:.0f}%")